In [1]:
import re
import regex
assert "regex" in str(regex.compile("(test){e<=1}"))
import pandas as pd
import os
import numpy as np
pd.set_option('display.max_colwidth', 80)
from nltk.corpus import stopwords
from sklearn.model_selection import (train_test_split, StratifiedKFold,
cross_val_predict)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from pathlib import Path
project_dir = Path().resolve()
os.chdir(project_dir)

In [2]:
df = pd.read_csv('hand_coded_sample_post_cor.csv')
print(len(df))
for col in ["complaint_clean", "findings_clean"]:
    df[col] = (
        df[col]
        .str.replace(r"\bcpoa\b", "agency", regex=True, case=False)
        .str.replace(r"\bapd\b", "police", regex=True, case=False)
    )
df['snippet_clean'] = (
    df["complaint_clean"].fillna("").str.strip()
    + "\n"
    + df["findings_clean"].fillna("").str.strip()
).str.strip()

190


In [3]:
df[[
    "mental_health",
    "regulatory_complaint",
    "demeanor_complaint",
    "bias_complaint",
    "traffic_scenario",
    "property_complaint"
]].apply(pd.Series.value_counts)

,mental_health,regulatory_complaint,demeanor_complaint,bias_complaint,traffic_scenario,property_complaint
0,180,154,120,179,156,176
1,10,36,70,11,34,14


In [4]:
vectorizers = {
    "tfidf": TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.80
    ),

    "sbert": SentenceTransformer("all-MiniLM-L6-v2")
}

models = {
    "nb_bal": MultinomialNB(class_prior=[0.5,0.5]
    ),

    "logreg_2k_bal": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=123
    ),

    "logreg_1k_bal": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=123
    ),

    "svm_lin": LinearSVC(
        class_weight="balanced",
        random_state=123
    ),

    "rf": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=123,
        n_jobs=-1
    ),

    "svm": SVC(
    kernel="linear",
    class_weight="balanced",
    random_state=123
)

}

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## MH SNIPPET

In [5]:
TARGET = "mental_health"
TEXT_COLUMN = "snippet_clean"

In [6]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print(confusion_matrix(y_test, y_pred))

Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

[[36  0]
 [ 2  0]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

[[36  0]
 [ 2  0]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_1k_bal
   

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

[[36  0]
 [ 2  0]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

[[36  0]
 [ 2  0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : mental_health
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        36
           1       0.00      0.00      0.00         2

    accuracy                           0.89        38
   macro avg       0.47      0.47      0.47        38
weighted avg       0.89      0.89      0.89        38

[[34  2]
 [ 2  0]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        36
           1       0.00      0.00      0.00         2

    accuracy                           0.89        38
   macro avg       0.47      0.47      0.47        38
weighted avg       0.89      0.89      0.89        38

[[34  2]
 [ 2  0]]
Target      : mental_health
Text        : snippet_clean
Vectorizer  : sbert
Model       : svm_lin
  

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## MH COMPLAINT

In [7]:
TARGET = "mental_health"
TEXT_COLUMN = "complaint_clean"

In [8]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : mental_health
Text        : complaint_clean
Vectorizer 

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : mental_health
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        36
           1       0.50      0.50      0.50         2

    accuracy                           0.95        38
   macro avg       0.74      0.74      0.74        38
weighted avg       0.95      0.95      0.95        38

Confusion Matrix
[[35  1]
 [ 1  1]]
Target      : mental_health
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.97      0.97      0.97        36
           1       0.50      0.50      0.50         2

    accuracy                           0.95        38
   macro avg       0.74      0.74      0.74        38
weighted avg       0.95      0.95      0.95        38

Confusion Matrix
[[35  1]
 [ 1  1]]
Target      : mental_health
Text        : complaint_clean
Vect

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## BIAS SNIPPET

In [9]:
TARGET = "bias_complaint"
TEXT_COLUMN = "snippet_clean"

In [10]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : 

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       1.00      1.00      1.00         2

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

Confusion Matrix
[[36  0]
 [ 0  2]]
Target      : bias_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        36
           1       1.00      1.00      1.00         2

    accuracy                           1.00        38
   macro avg       1.00      1.00      1.00        38
weighted avg       1.00      1.00      1.00        38

Confusion Matrix
[[36  0]
 [ 0  2]]
Target      : bias_complaint
Text        : snippet_clean
Vectori

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## BIAS COMPLAINT

In [11]:
TARGET = "bias_complaint"
TEXT_COLUMN = "complaint_clean"

In [12]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : bias_complaint
Text        : complaint_clean
Vectoriz

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        36
           1       0.00      0.00      0.00         2

    accuracy                           0.95        38
   macro avg       0.47      0.50      0.49        38
weighted avg       0.90      0.95      0.92        38

Confusion Matrix
[[36  0]
 [ 2  0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.97      1.00      0.99        36
           1       1.00      0.50      0.67         2

    accuracy                           0.97        38
   macro avg       0.99      0.75      0.83        38
weighted avg       0.97      0.97      0.97        38

Confusion Matrix
[[36  0]
 [ 1  1]]
Target      : bias_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.97      1.00      0.99        36
           1       1.00      0.50      0.67         2

    accuracy                           0.97        38
   macro avg       0.99      0.75      0.83        38
weighted avg       0.97      0.97      0.97        38

Confusion Matrix
[[36  0]
 [ 1  1]]
Target      : bias_complaint
Text        : complaint_clean
V

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## PROPERTY SNIPPET

In [13]:
TARGET = "property_complaint"
TEXT_COLUMN = "snippet_clean"

In [14]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.92      0.97      0.94        35
           1       0.00      0.00      0.00         3

    accuracy                           0.89        38
   macro avg       0.46      0.49      0.47        38
weighted avg       0.85      0.89      0.87        38

Confusion Matrix
[[34  1]
 [ 3  0]]
Target      : property_complaint
Text        : snippet_clean
Ve

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : property_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        35
           1       0.43      1.00      0.60         3

    accuracy                           0.89        38
   macro avg       0.71      0.94      0.77        38
weighted avg       0.95      0.89      0.91        38

Confusion Matrix
[[31  4]
 [ 0  3]]
Target      : property_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        35
           1       0.43      1.00      0.60         3

    accuracy                           0.89        38
   macro avg       0.71      0.94      0.77        38
weighted avg       0.95      0.89      0.91        38

Confusion Matrix
[[31  4]
 [ 0  3]]
Target      : property_complaint
Text        : snippet_c

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## PROPERTY COMPLAINT

In [15]:
TARGET = "property_complaint"
TEXT_COLUMN = "complaint_clean"

In [16]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]
Target      : property_complaint
Text        : complaint_cl

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       0.00      0.00      0.00         3

    accuracy                           0.92        38
   macro avg       0.46      0.50      0.48        38
weighted avg       0.85      0.92      0.88        38

Confusion Matrix
[[35  0]
 [ 3  0]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : property_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.94      0.91      0.93        35
           1       0.25      0.33      0.29         3

    accuracy                           0.87        38
   macro avg       0.60      0.62      0.61        38
weighted avg       0.89      0.87      0.88        38

Confusion Matrix
[[32  3]
 [ 2  1]]
Target      : property_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.94      0.91      0.93        35
           1       0.25      0.33      0.29         3

    accuracy                           0.87        38
   macro avg       0.60      0.62      0.61        38
weighted avg       0.89      0.87      0.88        38

Confusion Matrix
[[32  3]
 [ 2  1]]
Target      : property_complaint
Text        : compl

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## DEMEANOR SNIPPET

In [17]:
TARGET = "demeanor_complaint"
TEXT_COLUMN = "snippet_clean"

In [18]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.66      0.96      0.78        24
           1       0.67      0.14      0.24        14

    accuracy                           0.66        38
   macro avg       0.66      0.55      0.51        38
weighted avg       0.66      0.66      0.58        38

Confusion Matrix
[[23  1]
 [12  2]]
Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.69      0.83      0.75        24
           1       0.56      0.36      0.43        14

    accuracy                           0.66        38
   macro avg       0.62      0.60      0.59        38
weighted avg       0.64      0.66      0.64        38

Confusion Matrix
[[20  4]
 [ 9  5]]
Target      : demeanor_complaint
Text        : snippet_clean
Ve

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.73      0.67      0.70        24
           1       0.50      0.57      0.53        14

    accuracy                           0.63        38
   macro avg       0.61      0.62      0.61        38
weighted avg       0.64      0.63      0.64        38

Confusion Matrix
[[16  8]
 [ 6  8]]
Target      : demeanor_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.73      0.67      0.70        24
           1       0.50      0.57      0.53        14

    accuracy                           0.63        38
   macro avg       0.61      0.62      0.61        38
weighted avg       0.64      0.63      0.64        38

Confusion Matrix
[[16  8]
 [ 6  8]]
Target      : demeanor_complaint
Text        : snippet_c

## DEMEANOR COMPLAINT

In [19]:
TARGET = "demeanor_complaint"
TEXT_COLUMN = "complaint_clean"

In [20]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.68      0.79      0.73        24
           1       0.50      0.36      0.42        14

    accuracy                           0.63        38
   macro avg       0.59      0.57      0.57        38
weighted avg       0.61      0.63      0.62        38

Confusion Matrix
[[19  5]
 [ 9  5]]
Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.77      0.71      0.74        24
           1       0.56      0.64      0.60        14

    accuracy                           0.68        38
   macro avg       0.67      0.68      0.67        38
weighted avg       0.70      0.68      0.69        38

Confusion Matrix
[[17  7]
 [ 5  9]]
Target      : demeanor_complaint
Text        : complaint_cl

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.75      0.75      0.75        24
           1       0.57      0.57      0.57        14

    accuracy                           0.68        38
   macro avg       0.66      0.66      0.66        38
weighted avg       0.68      0.68      0.68        38

Confusion Matrix
[[18  6]
 [ 6  8]]
Target      : demeanor_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.75      0.75      0.75        24
           1       0.57      0.57      0.57        14

    accuracy                           0.68        38
   macro avg       0.66      0.66      0.66        38
weighted avg       0.68      0.68      0.68        38

Confusion Matrix
[[18  6]
 [ 6  8]]
Target      : demeanor_complaint
Text        : compl

## REGULATORY SNIPPET

In [21]:
TARGET = "regulatory_complaint"
TEXT_COLUMN = "snippet_clean"

In [22]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.84      1.00      0.91        31
           1       1.00      0.14      0.25         7

    accuracy                           0.84        38
   macro avg       0.92      0.57      0.58        38
weighted avg       0.87      0.84      0.79        38

Confusion Matrix
[[31  0]
 [ 6  1]]
Target      : regulatory_complaint
Text        : snippet_cl

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.84      1.00      0.91        31
           1       1.00      0.14      0.25         7

    accuracy                           0.84        38
   macro avg       0.92      0.57      0.58        38
weighted avg       0.87      0.84      0.79        38

Confusion Matrix
[[31  0]
 [ 6  1]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.82      0.74      0.78        31
           1       0.20      0.29      0.24         7

    accuracy                           0.66        38
   macro avg       0.51      0.51      0.51        38
weighted avg       0.71      0.66      0.68        38

Confusion Matrix
[[23  8]
 [ 5  2]]
Target      : regulatory_complaint
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.82      0.74      0.78        31
           1       0.20      0.29      0.24         7

    accuracy                           0.66        38
   macro avg       0.51      0.51      0.51        38
weighted avg       0.71      0.66      0.68        38

Confusion Matrix
[[23  8]
 [ 5  2]]
Target      : regulatory_complaint
Text        : sni

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## REGULATORY COMPLAINT

In [23]:
TARGET = "regulatory_complaint"
TEXT_COLUMN = "complaint_clean"

In [24]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.83      0.97      0.90        31
           1       0.50      0.14      0.22         7

    accuracy                           0.82        38
   macro avg       0.67      0.56      0.56        38
weighted avg       0.77      0.82      0.77        38

Confusion Matrix
[[30  1]
 [ 6  1]]
Target      : regulatory_complaint
Text        : compla

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.83      0.97      0.90        31
           1       0.50      0.14      0.22         7

    accuracy                           0.82        38
   macro avg       0.67      0.56      0.56        38
weighted avg       0.77      0.82      0.77        38

Confusion Matrix
[[30  1]
 [ 6  1]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.89      0.81      0.85        31
           1       0.40      0.57      0.47         7

    accuracy                           0.76        38
   macro avg       0.65      0.69      0.66        38
weighted avg       0.80      0.76      0.78        38

Confusion Matrix
[[25  6]
 [ 3  4]]
Target      : regulatory_complaint
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.89      0.81      0.85        31
           1       0.40      0.57      0.47         7

    accuracy                           0.76        38
   macro avg       0.65      0.69      0.66        38
weighted avg       0.80      0.76      0.78        38

Confusion Matrix
[[25  6]
 [ 3  4]]
Target      : regulatory_complaint
Text        :

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

## TRAFFIC SNIPPET

In [25]:
TARGET = "traffic_scenario"
TEXT_COLUMN = "snippet_clean"

In [26]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        31
           1       1.00      0.57      0.73         7

    accuracy                           0.92        38
   macro avg       0.96      0.79      0.84        38
weighted avg       0.93      0.92      0.91        38

Confusion Matrix
[[31  0]
 [ 3  4]]
Target      : traffic_scenario
Text        : snippet_clean
Vectoriz

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.91      1.00      0.95        31
           1       1.00      0.57      0.73         7

    accuracy                           0.92        38
   macro avg       0.96      0.79      0.84        38
weighted avg       0.93      0.92      0.91        38

Confusion Matrix
[[31  0]
 [ 3  4]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        31
           1       0.71      0.71      0.71         7

    accuracy                           0.89        38
   macro avg       0.82      0.82      0.82        38
weighted avg       0.89      0.89      0.89        38

Confusion Matrix
[[29  2]
 [ 2  5]]
Target      : traffic_scenario
Text        : snippet_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        31
           1       0.71      0.71      0.71         7

    accuracy                           0.89        38
   macro avg       0.82      0.82      0.82        38
weighted avg       0.89      0.89      0.89        38

Confusion Matrix
[[29  2]
 [ 2  5]]
Target      : traffic_scenario
Text        : snippet_clean
V

## TRAFFIC COMPLAINT

In [27]:
TARGET = "traffic_scenario"
TEXT_COLUMN = "complaint_clean"

In [28]:
df_train = df.dropna(subset=[TARGET])

X = df_train[TEXT_COLUMN]
y = df_train[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

for vectorizer_name, vectorizer in vectorizers.items():
    if vectorizer_name == "tfidf":
        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

    elif vectorizer_name == "sbert":
        X_train_vec = vectorizer.encode(
            X_train.tolist(),
            show_progress_bar=True)
        X_test_vec = vectorizer.encode(
            X_test.tolist(),
            show_progress_bar=True)
    for model_name, clf in models.items():
        if vectorizer_name == "sbert" and model_name in ["nb_bal", "nb_unbal"]:
            continue
        clf.fit(X_train_vec, y_train)

        y_pred = clf.predict(X_test_vec)

        print("=" * 70)
        print(f"Target      : {TARGET}")
        print(f"Text        : {TEXT_COLUMN}")
        print(f"Vectorizer  : {vectorizer_name}")
        print(f"Model       : {model_name}")
        print("=" * 70)

        print(classification_report(y_test, y_pred))
        print("Confusion Matrix")
        print(confusion_matrix(y_test, y_pred))

Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model       : nb_bal
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.87      0.87      0.87        31
           1       0.43      0.43      0.43         7

    accuracy                           0.79        38
   macro avg       0.65      0.65      0.65        38
weighted avg       0.79      0.79      0.79        38

Confusion Matrix
[[27  4]
 [ 4  3]]
Target      : traffic_scenario
Text        : complaint_clean
Ve

/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model       : rf
              precision    recall  f1-score   support

           0       0.82      1.00      0.90        31
           1       0.00      0.00      0.00         7

    accuracy                           0.82        38
   macro avg       0.41      0.50      0.45        38
weighted avg       0.67      0.82      0.73        38

Confusion Matrix
[[31  0]
 [ 7  0]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : tfidf
Model       : svm
              precision    recall  f1-score   support

           0       0.84      0.87      0.86        31
           1       0.33      0.29      0.31         7

    accuracy                           0.76        38
   macro avg       0.59      0.58      0.58        38
weighted avg       0.75      0.76      0.76        38

Confusion Matrix
[[27  4]
 [ 5  2]]


/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/michelleupham/Desktop/Programs/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_2k_bal
              precision    recall  f1-score   support

           0       0.93      0.87      0.90        31
           1       0.56      0.71      0.62         7

    accuracy                           0.84        38
   macro avg       0.74      0.79      0.76        38
weighted avg       0.86      0.84      0.85        38

Confusion Matrix
[[27  4]
 [ 2  5]]
Target      : traffic_scenario
Text        : complaint_clean
Vectorizer  : sbert
Model       : logreg_1k_bal
              precision    recall  f1-score   support

           0       0.93      0.87      0.90        31
           1       0.56      0.71      0.62         7

    accuracy                           0.84        38
   macro avg       0.74      0.79      0.76        38
weighted avg       0.86      0.84      0.85        38

Confusion Matrix
[[27  4]
 [ 2  5]]
Target      : traffic_scenario
Text        : complaint_c